# SpaceX Falcon 9 First Stage Landing Prediction
## Interactive Geospatial Visual Analytics with Folium
**Author:** Dhruv Khokhriya  
**Course:** IBM Applied Data Science Capstone  

### Objectives
- Plot all active launch site coordinates (`CCAFS LC-40`, `CCAFS SLC-40`, `KSC LC-39A`, `VAFB SLC-4E`) on an interactive Folium Map.
- Deploy `folium.CircleMarker` badges color-coded by landing outcome (Green = Success, Red = Failure).
- Cluster launch records using `MarkerCluster` to manage visual density across zoom levels.
- Perform geodesic proximity calculations from launch pads to coastlines, railways, and highways using `folium.PolyLine`.

In [ ]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from math import sin, cos, sqrt, atan2, radians

df = pd.read_csv('spacex_data.csv')
print("Loaded coordinates for", len(df), "missions.")

### 1. Build Base Folium Map Centered at Cape Canaveral

In [ ]:
site_map = folium.Map(location=[28.5623, -80.5774], zoom_start=5)

# Add site markers
launch_sites = df[['LaunchSite', 'Latitude', 'Longitude']].drop_duplicates()
for _, row in launch_sites.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=row['LaunchSite'],
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(site_map)

site_map

### 2. Add Color-Coded MarkerCluster for Landing Outcomes

In [ ]:
marker_cluster = MarkerCluster().add_to(site_map)

for _, row in df.iterrows():
    color = 'green' if row['Class'] == 1 else 'red'
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Flight {row['FlightNumber']}: {row['Outcome']}",
        icon=folium.Icon(color=color, icon='ok-sign' if row['Class'] == 1 else 'remove-sign')
    ).add_to(marker_cluster)

site_map

### 3. Calculate Geodesic Distance to Coastline and Infrastructure

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Distance from KSC LC-39A to coastline
ksc_lat, ksc_lon = 28.573255, -80.646895
coast_lat, coast_lon = 28.573255, -80.5716
dist_coast = calculate_distance(ksc_lat, ksc_lon, coast_lat, coast_lon)
print(f"Calculated geodesic distance from KSC LC-39A to Coastline: {dist_coast:.2f} km")